# CNN Training and Validation for Spot Detection

Train and evaluate CNN models for fluorescent spot detection.

## Workflow
```
1. Generate_Training_Crops.ipynb
   └── Creates: training/training_crops_*/

2. Human_Labeling_Tool.ipynb
   └── Labels crops → training/human_selection_*.npy

3. ML_Tranining_Validation.ipynb  ← YOU ARE HERE
   └── Trains models → spot_detection_cnn.pth
```

## Quick Start
| Mode | Setting | Time |
|------|---------|------|
| Evaluate only | `RETRAIN_MODELS = False` | ~1 min |
| Retrain models | `RETRAIN_MODELS = True` | ~30 min |

## Data Location
All training data is in `training/` folder:
- `training/training_crops_*/` - Image crops
- `training/human_selection_*.npy` - Annotations
- `training/*.pth` - Experimental models

## Deployed Model
| File | Location |
|------|----------|
| `spot_detection_cnn.pth` | Root folder (used by MicroLive GUI) |

In [1]:
"""
MicroLive Notebook
==================
This notebook requires MicroLive to be installed:
    pip install microlive

For development mode:
    pip install -e /path/to/microlive
"""
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Verify GPU support
check_gpu_status()

# Standard scientific imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


MicroLive root: /Users/nzlab-la/Desktop/microlive
Modules loaded successfully


## Configuration

In [2]:
# =============================================================================
# CONFIGURATION
# =============================================================================

RETRAIN_MODELS = False  # Set True to retrain, False to load existing

# Training hyperparameters
BATCH_SIZE = 256
NUM_EPOCHS = BATCH_SIZE * 200  # 51,200 epochs
LEARNING_RATE = 1e-6

# Paths to training data (in training/ folder)
CROPS_REAL = Path('training/training_crops_real_data')
CROPS_SIMULATED = Path('training/training_crops_simulated_data')
CROPS_HUMAN = Path('training/training_crops_human_selection')

# Model paths
MODEL_REAL = Path('training/particle_detection_cnn_real_data.pth')
MODEL_SIMULATED = Path('training/particle_detection_cnn_simulated_data.pth')
MODEL_HUMAN = Path('spot_detection_cnn.pth')  # Deployed in MicroLive (root folder)

# Verify files
print("Model files:")
for name, path in [("Real", MODEL_REAL), ("Simulated", MODEL_SIMULATED), ("Human", MODEL_HUMAN)]:
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {path}")

print("\nCrop folders:")
for name, folder in [("Real", CROPS_REAL), ("Simulated", CROPS_SIMULATED), ("Human", CROPS_HUMAN)]:
    if folder.exists():
        n = len(list(folder.glob('*.png')))
        print(f"  ✓ {name}: {n} crops")
    else:
        print(f"  ✗ {name}: NOT FOUND")

print(f"\nRETRAIN_MODELS = {RETRAIN_MODELS}")

Model files:
  ✓ training/particle_detection_cnn_real_data.pth
  ✓ training/particle_detection_cnn_simulated_data.pth
  ✓ spot_detection_cnn.pth

Crop folders:
  ✓ Real: 2000 crops
  ✓ Simulated: 6000 crops
  ✓ Human: 120 crops

RETRAIN_MODELS = False


## 1. Model: Real Data

In [3]:
if RETRAIN_MODELS:
    print("Training on real data crops...")
    model_real, losses_real_train, losses_real_val = ML.run_network(
        image_dir=str(CROPS_REAL),
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE
    )
    ML.save_model(model_real, path=str(MODEL_REAL))
    
    plt.figure(figsize=(8, 4))
    plt.plot(losses_real_train, label='Train', alpha=0.7)
    plt.plot(losses_real_val, label='Validation', alpha=0.7)
    plt.title('Real Data Model')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print(f"Loading: {MODEL_REAL}")
    model_real = ML.ParticleDetectionCNN()
    ML.load_model(model_real, str(MODEL_REAL))
    print("Loaded successfully")

Loading: training/particle_detection_cnn_real_data.pth
Loaded successfully


## 2. Model: Simulated Data

In [4]:
if RETRAIN_MODELS:
    print("Training on simulated data crops...")
    model_simulated, losses_sim_train, losses_sim_val = ML.run_network(
        image_dir=str(CROPS_SIMULATED),
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE
    )
    ML.save_model(model_simulated, path=str(MODEL_SIMULATED))
    
    plt.figure(figsize=(8, 4))
    plt.plot(losses_sim_train, label='Train', alpha=0.7)
    plt.plot(losses_sim_val, label='Validation', alpha=0.7)
    plt.title('Simulated Data Model')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print(f"Loading: {MODEL_SIMULATED}")
    model_simulated = ML.ParticleDetectionCNN()
    ML.load_model(model_simulated, str(MODEL_SIMULATED))
    print("Loaded successfully")

Loading: training/particle_detection_cnn_simulated_data.pth
Loaded successfully


## 3. Model: Human-Curated ⭐ (Deployed in MicroLive)

In [5]:
if RETRAIN_MODELS:
    print("Training on human-curated consensus crops...")
    model_human, losses_human_train, losses_human_val = ML.run_network(
        image_dir=str(CROPS_HUMAN),
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE
    )
    ML.save_model(model_human, path=str(MODEL_HUMAN))
    
    plt.figure(figsize=(8, 4))
    plt.plot(losses_human_train, label='Train', alpha=0.7)
    plt.plot(losses_human_val, label='Validation', alpha=0.7)
    plt.title('Human-Curated Model (Deployed)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print(f"Loading: {MODEL_HUMAN}")
    model_human = ML.ParticleDetectionCNN()
    ML.load_model(model_human, str(MODEL_HUMAN))
    print("Loaded successfully")

Loading: spot_detection_cnn.pth
Loaded successfully


## 4. Evaluation

Load human annotations and compare model predictions.

In [6]:
# Load human annotations for validation
annotations = {}
for i in range(1, 4):
    path = Path(f'training/human_selection_{i}.npy')
    if path.exists():
        annotations[i] = np.load(path)
        print(f"Annotator {i}: {np.sum(annotations[i])} positives out of {len(annotations[i])}")

# Create consensus labels (all annotators agree)
if len(annotations) >= 3:
    a, b, c = annotations[1], annotations[2], annotations[3]
    consensus_positive = a & b & c
    consensus_negative = ~a & ~b & ~c
    ground_truth = consensus_positive  # Use all-agree-positive as ground truth
    
    print(f"\nConsensus:")
    print(f"  All agree positive: {np.sum(consensus_positive)}")
    print(f"  All agree negative: {np.sum(consensus_negative)}")
    print(f"  Ambiguous: {len(a) - np.sum(consensus_positive) - np.sum(consensus_negative)}")

Annotator 1: 70 positives out of 149
Annotator 2: 83 positives out of 149
Annotator 3: 95 positives out of 149

Consensus:
  All agree positive: 69
  All agree negative: 51
  Ambiguous: 29


In [7]:
def calculate_metrics(predicted, ground_truth):
    """Calculate classification metrics."""
    TP = np.sum((predicted == True) & (ground_truth == True))
    FP = np.sum((predicted == True) & (ground_truth == False))
    TN = np.sum((predicted == False) & (ground_truth == False))
    FN = np.sum((predicted == False) & (ground_truth == True))
    
    accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1
    }

def print_metrics(name, metrics):
    """Print metrics in a formatted way."""
    print(f"{name}:")
    print(f"  TP={metrics['TP']}, FP={metrics['FP']}, TN={metrics['TN']}, FN={metrics['FN']}")
    print(f"  Accuracy: {metrics['Accuracy']:.1%}")
    print(f"  Precision: {metrics['Precision']:.1%}")
    print(f"  Recall: {metrics['Recall']:.1%}")
    print(f"  F1: {metrics['F1']:.2f}")

## 5. Compare Models

Evaluate each model on the same test set.

In [8]:
# Load test crops from human-curated folder
test_crops = []
test_labels = []

particle_files = sorted(CROPS_HUMAN.glob('particle_*.png'))
no_particle_files = sorted(CROPS_HUMAN.glob('no_particle_*.png'))

for f in particle_files:
    img = np.array(Image.open(f).convert('L'))
    test_crops.append(img)
    test_labels.append(True)

for f in no_particle_files:
    img = np.array(Image.open(f).convert('L'))
    test_crops.append(img)
    test_labels.append(False)

test_labels = np.array(test_labels)
print(f"Test set: {len(test_crops)} crops ({np.sum(test_labels)} positive, {len(test_labels) - np.sum(test_labels)} negative)")

Test set: 120 crops (69 positive, 51 negative)


In [9]:
# Evaluate each model
results = {}
threshold = 0.51

for name, model in [("Real", model_real), ("Simulated", model_simulated), ("Human (Deployed)", model_human)]:
    # predict_crops returns (flag_vector, probability_values)
    predictions, probabilities = ML.predict_crops(model, test_crops, threshold=threshold)
    predictions = predictions.astype(bool)  # Convert to boolean for comparison
    metrics = calculate_metrics(predictions, test_labels)
    results[name] = metrics
    print_metrics(name, metrics)
    print(f"  Mean probability: {np.mean(probabilities):.3f}")
    print()

Real:
  TP=69, FP=42, TN=9, FN=0
  Accuracy: 65.0%
  Precision: 62.2%
  Recall: 100.0%
  F1: 0.77
  Mean probability: 0.685

Simulated:
  TP=69, FP=49, TN=2, FN=0
  Accuracy: 59.2%
  Precision: 58.5%
  Recall: 100.0%
  F1: 0.74
  Mean probability: 0.699

Human (Deployed):
  TP=68, FP=2, TN=49, FN=1
  Accuracy: 97.5%
  Precision: 97.1%
  Recall: 98.6%
  F1: 0.98
  Mean probability: 0.633



## Summary

In [10]:
# Create summary DataFrame
summary_data = []
for name, metrics in results.items():
    summary_data.append({
        'Model': name,
        'Accuracy': f"{metrics['Accuracy']:.1%}",
        'Precision': f"{metrics['Precision']:.1%}",
        'Recall': f"{metrics['Recall']:.1%}",
        'F1': f"{metrics['F1']:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

           Model Accuracy Precision Recall   F1
            Real    65.0%     62.2% 100.0% 0.77
       Simulated    59.2%     58.5% 100.0% 0.74
Human (Deployed)    97.5%     97.1%  98.6% 0.98
